In [50]:
from lcpy.calculators.bw_int import mpLCAer
import os
from lcpy.hvs.hvs import store_scenario_results
from lcpy.calculators.env_calc import fast_calculator
import pandas as pd
from lcpy.hvs.map_dicts import create_mapping, create_list_with_unique_activities
import numpy as np

The initial configuration remains the same with the simple LCA example

In [51]:
target_dir = "results/dummy_results"
os.makedirs(target_dir, exist_ok=True)

In [52]:
brightway_configuration_dictionary = {
    "path_to_brightway_project": "/Users/tsuitpy/Library/Application Support/Brightway3/dummy_project.f8a300edbb1471040f4e54648fcf5b74",
    "bw_project": "dummy_project",
    "bw_database": "dummy_project_database", 
    "bw_biosphere": "ecoinvent-3.9.1-biosphere",
    "bw_ecoinvent": "ecoinvent-3.9.1-cutoff"
}

In [53]:
methods_list = [
('TRACI v2.1', 'acidification', 'acidification potential (AP)'),
('TRACI v2.1', 'climate change', 'global warming potential (GWP100)'),
('TRACI v2.1', 'ecotoxicity: freshwater', 'ecotoxicity: freshwater'),
('TRACI v2.1', 'eutrophication', 'eutrophication potential'),
('TRACI v2.1', 'human toxicity: carcinogenic', 'human toxicity: carcinogenic'),
('TRACI v2.1', 'human toxicity: non-carcinogenic', 'human toxicity: non-carcinogenic'),
('TRACI v2.1', 'ozone depletion', 'ozone depletion potential (ODP)'),
('TRACI v2.1', 'particulate matter formation', 'particulate matter formation potential (PMFP)'),
('TRACI v2.1', 'photochemical oxidant formation', 'maximum incremental reactivity (MIR)'),
]

method_units_list = ['kg SO2-Eq',
 'kg CO2-Eq',
 'CTUe',
 'kg N-Eq',
 'CTUh',
 'CTUh',
 'kg CFC-11-Eq',
 'kg PM2.5-Eq',
 'kg O3-Eq',
]

In [54]:
methods_gp = methods_list[:]
impact_categories_names = ['AP', 'GWP100', 'ECFW', 'EP', 'HTC', 'HTNC', 'ODP', 'PMFP', 'MIR']

In contrast to the simple LCA we now set the number of scenarios to a high number that basically represents the MC loops

In [55]:
scenarios = 100
timeframe = 1 #operational lifetime after construction
time_step = 1
construction_years = 0
number_of_infrastructure_processes = 0

The parametric model should now be set to include uncertainty for the MC loops. This is done below, by setting a number of parameters to raneg with some probability distributions

In [56]:
keys_nuclear_power_generation = {
    'PWR': '8ce68e5f71f8f8c5dc50edb913e9cea1',
    'BWR': '8ce68e5f71f8f8c5dc50edb913e9cea1'
}

keys_fossil_power_generation = {
    'NG_ccpp': '8ce68e5f71f8f8c5dc50edb913e9cea1',
    'NG_convpp': '8ce68e5f71f8f8c5dc50edb913e9cea1',
    'NG_cogen_conv': '8ce68e5f71f8f8c5dc50edb913e9cea1',
    'NG_cogen_cc': '8ce68e5f71f8f8c5dc50edb913e9cea1',
}

keys_res_power_generation = {
    'Hydro': '8ce68e5f71f8f8c5dc50edb913e9cea1',
    'DGE': '8ce68e5f71f8f8c5dc50edb913e9cea1',
    'Wind': '8ce68e5f71f8f8c5dc50edb913e9cea1',
}

keys_total_power_generation = {
    'Nuclear': '',
    'Fossil': '',
    'RES': '',
}

In [57]:
key_list_sub_processes = [keys_nuclear_power_generation, keys_fossil_power_generation, keys_res_power_generation]
mapping_names = create_mapping(keys_total_power_generation, key_list_sub_processes)
unique_activities = create_list_with_unique_activities(key_list_sub_processes)
my_lca = mpLCAer(4, methods_gp, brightway_configuration_dictionary)
my_lca.import_isolated_environment()
my_lca.lca_calculations(mapping_names)

In [58]:
def model(BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity,
     Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity):

    nuclear_capacity = BWR_capacity + PWR_capacity
    ng_capacity = NG_ccpp_capacity + NG_convpp_capacity + NG_cogen_conv_capacity + NG_cogen_cc_capacity
    res_capacity = Hydro_pumped_capacity + DGE_capacity + Wind_1_3_capacity
    total_capacity = nuclear_capacity + ng_capacity + res_capacity

    nuclear_exchanges_amounts = [
        PWR_capacity / nuclear_capacity,
        BWR_capacity / nuclear_capacity
    ]

    fossil_exchanges_amounts = [
        NG_ccpp_capacity / ng_capacity,
        NG_convpp_capacity / ng_capacity,
        NG_cogen_conv_capacity / ng_capacity,
        NG_cogen_cc_capacity / ng_capacity,
    ]

    res_exchanges_amounts = [
        Hydro_pumped_capacity / res_capacity,
        DGE_capacity / res_capacity,
        Wind_1_3_capacity / res_capacity,
    ]

    mp_exchanges_amounts = [
        nuclear_capacity / total_capacity,
        ng_capacity / total_capacity,
        res_capacity / total_capacity,
    ]
    exchanges_list_sp = [nuclear_exchanges_amounts, fossil_exchanges_amounts, res_exchanges_amounts]
    mapping_exchanges = create_mapping(keys_total_power_generation, exchanges_list_sp)
    my_calculator = fast_calculator()

    my_calculator.calculation_static_lcia(mapping_exchanges, my_lca.unit_impacts)
    my_calculator.calculation_impact_senarios(mp_exchanges_amounts, my_calculator.impact, 'Electricity production')

    gsa_result = my_calculator.total_impact['Electricity production'][:,:,0].T

    return gsa_result

In [66]:
test_result = model(
    np.array([[1]]), np.array([[2]]), np.array([[3]]),
    np.array([[4]]), np.array([[5]]), np.array([[6]]),
    np.array([[7]]), np.array([[8]]), np.array([[9]])
)

In [59]:
from pymoo.core.problem import ElementwiseProblem, Problem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

In [ ]:
low_bounds = np.array([1200, 700, 350, 600, 560, 230, 320, 55, 900])
high_bounds = np.array([1800, 900, 450, 800, 700, 280, 579, 115, 1100])

In [61]:
class Single(ElementwiseProblem):
    def __init__(self):
        super().__init__(
            n_var=9,
            n_obj=1,
            xl = low_bounds,
            xu = high_bounds,
            elementwise_evaluation=True
        )

    def _evaluate(self, x, out, *args, **kwargs):

        BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity, Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity = [np.array([value]).reshape((1,1)) for value in x]

        F = model(BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity, Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity)
        # If your model returns shape (1,1) for a single objective:
        out["F"] = F[0, 1]

In [62]:
problem = Single()
algorithm = GA(pop_size=100)
res = minimize(
    problem,
    algorithm,
    ('n_gen', 200),
    seed=42,
    verbose=True
)

n_gen  |  n_eval  |     f_avg     |     f_min    
     1 |      100 |  1.6530508764 |  1.6530508764
     2 |      200 |  1.6530508764 |  1.6530508764
     3 |      300 |  1.6530508764 |  1.6530508764
     4 |      400 |  1.6530508764 |  1.6530508764
     5 |      500 |  1.6530508764 |  1.6530508764
     6 |      600 |  1.6530508764 |  1.6530508764
     7 |      700 |  1.6530508764 |  1.6530508764
     8 |      800 |  1.6530508764 |  1.6530508764
     9 |      900 |  1.6530508764 |  1.6530508764
    10 |     1000 |  1.6530508764 |  1.6530508764
    11 |     1100 |  1.6530508764 |  1.6530508764
    12 |     1200 |  1.6530508764 |  1.6530508764
    13 |     1300 |  1.6530508764 |  1.6530508764
    14 |     1400 |  1.6530508764 |  1.6530508764
    15 |     1500 |  1.6530508764 |  1.6530508764
    16 |     1600 |  1.6530508764 |  1.6530508764
    17 |     1700 |  1.6530508764 |  1.6530508764
    18 |     1800 |  1.6530508764 |  1.6530508764
    19 |     1900 |  1.6530508764 |  1.6530508764


In [63]:
class Multi(ElementwiseProblem):
    def __init__(self):
        super().__init__(
            n_var=9,
            n_obj=2,
            n_constr=0,
            xl=low_bounds,
            xu=high_bounds,
            elementwise_evaluation=True
        )

    def _evaluate(self, x, out, *args, **kwargs):

        BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity, Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity = [np.array([value]).reshape((1,1)) for value in x]

        F = model(BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity, Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity)

        out["F"] = F[0, :2]

In [64]:
mo_problem = Multi()
mo_algo    = NSGA2(pop_size=100)
mo_res     = minimize(
    mo_problem,
    mo_algo,
    ('n_gen', 200),
    seed=42,
    verbose=True
)

n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |      100 |      1 |             - |             -
     2 |      200 |      1 |  0.000000E+00 |             f
     3 |      300 |      1 |  0.000000E+00 |             f
     4 |      400 |      1 |  0.000000E+00 |             f
     5 |      500 |      1 |  0.000000E+00 |             f
     6 |      600 |      1 |  0.000000E+00 |             f
     7 |      700 |      2 |  0.000000E+00 |             f
     8 |      800 |      2 |  0.000000E+00 |             f
     9 |      900 |      2 |  0.000000E+00 |             f
    10 |     1000 |      3 |  0.000000E+00 |             f
    11 |     1100 |      3 |  0.000000E+00 |             f
    12 |     1200 |      5 |  0.000000E+00 |             f
    13 |     1300 |      5 |  0.000000E+00 |             f
    14 |     1400 |      5 |  0.000000E+00 |             f
    15 |     1500 |      6 |  0.000000E+00 |             f
    16 |     1600 |      6 |  0.000000E+00 |            

# Contributions:

- Show how to do it for the modular approach
- make it for emissions of an environmental flow
- change sampling strategies to allow for more probability distributions